In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06a_feature_correlation_reduction
# MAGIC 
# MAGIC **PARTE 1 DE 2**: Análisis de correlación y eliminación de features redundantes
# MAGIC 
# MAGIC **Objetivo**: 
# MAGIC - Analizar correlaciones entre features
# MAGIC - Eliminar variables altamente correlacionadas (|corr| > 0.85)
# MAGIC - Preparar datos para PCA (Parte 2)
# MAGIC 
# MAGIC **Input**: customer_features_rfm_20180930 (~70 features)
# MAGIC 
# MAGIC **Output**: Features seleccionadas listas para PCA

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
METRICS_PATH = "/Volumes/olist/olist_gold/metrics/"

print("=" * 80)
print("🚀 PARTE 1: REDUCCIÓN POR CORRELACIÓN")
print("=" * 80)
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Verificación del entorno

# COMMAND ----------

print("🔧 Verificando estructura de Unity Catalog...")
print()

# Crear catálogo si no existe
try:
    spark.sql("CREATE CATALOG IF NOT EXISTS olist")
    print("✅ Catálogo 'olist' verificado/creado")
except Exception as e:
    print(f"⚠️  Catálogo: {e}")

# Crear esquema olist_gold si no existe
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS olist.olist_gold")
    print("✅ Esquema 'olist.olist_gold' verificado/creado")
except Exception as e:
    print(f"⚠️  Esquema: {e}")

# Crear volumes si no existen
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.gold")
    print("✅ Volume 'gold' verificado/creado")
except Exception as e:
    print(f"⚠️  Volume gold: {e}")

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.metrics")
    print("✅ Volume 'metrics' verificado/creado")
except Exception as e:
    print(f"⚠️  Volume metrics: {e}")

print()
print("✅ Estructura de Unity Catalog lista")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Carga de datos

# COMMAND ----------

print("📥 Cargando features desde GOLD...")
source_path = f"{GOLD_PATH}customer_features_rfm_20180930/"
print(f"   Fuente: {source_path}")
print()

# Verificar que exista la tabla
try:
    files = dbutils.fs.ls(source_path)
    print(f"✅ Directorio encontrado con {len(files)} archivos")
except Exception as e:
    print(f"❌ ERROR: El directorio fuente no existe")
    print(f"   Ruta: {source_path}")
    print(f"   Error: {e}")
    print()
    print("⚠️  ACCIÓN REQUERIDA:")
    print("   Debes ejecutar primero el notebook de generación de features RFM")
    print("   (05_customer_features_engineering)")
    raise

# Cargar datos
try:
    df = spark.read.format("delta").load(source_path).toPandas()
    print(f"✅ Dataset cargado exitosamente")
    print(f"   - Registros: {len(df):,}")
    print(f"   - Columnas: {len(df.columns)}")
    print()
except Exception as e:
    print(f"❌ Error cargando datos: {e}")
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Preparación de datos

# COMMAND ----------

print("🎯 Separando features y target...")
print()

# Columnas a excluir de las features
exclude_cols = ["customer_id", "is_premium"]

# Si existe cluster_ordered, también excluirlo
if "cluster_ordered" in df.columns:
    exclude_cols.append("cluster_ordered")

# Identificar features
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Separar X y y
X_original = df[feature_cols].copy()
y = df["is_premium"].copy()
customer_id = df["customer_id"].copy()

print(f"✅ Separación completada:")
print(f"   - Features originales: {len(feature_cols)}")
print(f"   - Target: is_premium ({y.sum():,} premium / {len(y):,} total)")
print(f"   - Tasa premium: {y.mean()*100:.2f}%")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Manejo de valores faltantes y categóricos

# COMMAND ----------

print("🔧 Procesando features...")
print()

# Rellenar NaN con 0
X = X_original.fillna(0)
nan_count = X_original.isna().sum().sum()
if nan_count > 0:
    print(f"   ℹ️  Valores NaN rellenados con 0: {nan_count:,}")

# Convertir categóricas a dummies si existen
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_cols) > 0:
    print(f"   ℹ️  Variables categóricas encontradas: {len(categorical_cols)}")
    print(f"      {categorical_cols}")
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print(f"   ✅ Dummies creados. Nuevas features: {len(X.columns)}")
else:
    print(f"   ✅ No hay variables categóricas")

print(f"\n✅ Features finales para análisis: {len(X.columns)}")
print()

# Actualizar lista de features
feature_cols = X.columns.tolist()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Análisis de correlación

# COMMAND ----------

print("📊 ANÁLISIS DE CORRELACIÓN")
print("=" * 80)
print()

print("Calculando matriz de correlación...")
corr_matrix = X.corr()
print(f"✅ Matriz calculada: {corr_matrix.shape[0]} x {corr_matrix.shape[1]}")
print()

# Umbral de correlación
threshold = 0.85

print(f"Identificando pares con |correlación| > {threshold}...")
print()

# Identificar pares altamente correlacionados
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > threshold:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            high_corr_pairs.append((col1, col2, corr_val))

print(f"✅ Pares encontrados: {len(high_corr_pairs)}")
print()

if len(high_corr_pairs) > 0:
    print("Top 10 pares con mayor correlación:")
    sorted_pairs = sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)
    for col1, col2, corr in sorted_pairs[:10]:
        print(f"   • {col1:30s} <-> {col2:30s}: {corr:+.4f}")
    print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Eliminación de features correlacionadas

# COMMAND ----------

print("🗑️  Eliminando features redundantes...")
print("   Criterio: mantener variable alfabéticamente menor")
print()

# Determinar qué variables eliminar
to_drop = set()
for col1, col2, _ in high_corr_pairs:
    # Mantener la primera alfabéticamente
    if col1 < col2:
        to_drop.add(col2)
    else:
        to_drop.add(col1)

to_drop = sorted(list(to_drop))
features_retained = [c for c in feature_cols if c not in to_drop]

print(f"✅ Resumen:")
print(f"   - Features originales: {len(feature_cols)}")
print(f"   - Features eliminadas: {len(to_drop)}")
print(f"   - Features retenidas: {len(features_retained)}")
print(f"   - Reducción: {len(to_drop)/len(feature_cols)*100:.2f}%")
print()

# Aplicar reducción
X_reduced = X[features_retained].copy()

print(f"✅ Dataset reducido: {X_reduced.shape[0]:,} x {X_reduced.shape[1]}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Guardar features eliminadas

# COMMAND ----------

print("💾 Guardando features eliminadas...")
print()

if len(to_drop) > 0:
    dropped_df = pd.DataFrame({
        'feature': to_drop,
        'reason': 'correlation > 0.85'
    })
    
    # Guardar usando método temporal para CSV
    temp_path = f"{METRICS_PATH}features_temp/"
    spark.createDataFrame(dropped_df).write \
        .format("csv").mode("overwrite").option("header", "true") \
        .save(temp_path)
    
    # Mover el archivo CSV
    csv_files = [f for f in dbutils.fs.ls(temp_path) if f.name.endswith('.csv')]
    if csv_files:
        dbutils.fs.cp(csv_files[0].path, f"{METRICS_PATH}features_correlacion_eliminadas.csv")
    
    dbutils.fs.rm(temp_path, True)
    print(f"✅ Guardado: features_correlacion_eliminadas.csv")
    
    # Mostrar algunas features eliminadas
    print()
    print("Primeras 10 features eliminadas:")
    for feat in to_drop[:10]:
        print(f"   • {feat}")
    if len(to_drop) > 10:
        print(f"   ... y {len(to_drop)-10} más")
else:
    print("ℹ️  No hay features eliminadas por correlación")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Visualización de correlaciones

# COMMAND ----------

print("📈 Generando visualizaciones...")
print()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Histograma de correlaciones
ax1 = axes[0]
all_corrs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        all_corrs.append(abs(corr_matrix.iloc[i, j]))

ax1.hist(all_corrs, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold}')
ax1.set_xlabel('|Correlación|', fontsize=11)
ax1.set_ylabel('Frecuencia', fontsize=11)
ax1.set_title('Distribución de Correlaciones Absolutas', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Heatmap de correlación (muestra)
ax2 = axes[1]
if len(features_retained) > 25:
    # Muestra aleatoria
    sample_size = 25
    np.random.seed(42)
    sample_features = np.random.choice(features_retained, sample_size, replace=False)
    corr_sample = X_reduced[sample_features].corr()
    title = f'Matriz de Correlación\n(Muestra: {sample_size}/{len(features_retained)} features)'
else:
    corr_sample = X_reduced.corr()
    title = f'Matriz de Correlación\n({len(features_retained)} features retenidas)'

sns.heatmap(corr_sample, cmap='RdBu_r', center=0, ax=ax2,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, xticklabels=False, yticklabels=False)
ax2.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizaciones generadas")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Guardar dataset reducido (sin PCA aún)

# COMMAND ----------

print("💾 Guardando dataset intermedio...")
print()

# Crear DataFrame con features reducidas + customer_id + target
df_reduced = X_reduced.copy()
df_reduced['customer_id'] = customer_id.values
df_reduced['is_premium'] = y.values

# Reordenar columnas
cols_order = ['customer_id', 'is_premium'] + features_retained
df_reduced = df_reduced[cols_order]

# Guardar
intermediate_path = f"{GOLD_PATH}customer_features_rfm_20180930_after_correlation/"
spark.createDataFrame(df_reduced) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(intermediate_path)

print(f"✅ Dataset intermedio guardado:")
print(f"   Ubicación: {intermediate_path}")
print(f"   Shape: {df_reduced.shape}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Resumen de Parte 1

# COMMAND ----------

print()
print("=" * 80)
print("✅ PARTE 1 COMPLETADA: REDUCCIÓN POR CORRELACIÓN")
print("=" * 80)
print()

print("📊 RESUMEN:")
print("-" * 80)
print(f"Features originales:              {len(feature_cols):>6}")
print(f"Pares correlacionados (>{threshold}):    {len(high_corr_pairs):>6}")
print(f"Features eliminadas:              {len(to_drop):>6}")
print(f"Features retenidas:               {len(features_retained):>6}")
print(f"Reducción:                        {len(to_drop)/len(feature_cols)*100:>6.2f}%")
print()

print("💾 OUTPUTS GENERADOS:")
print("-" * 80)
print(f"✓ Dataset intermedio (Delta):")
print(f"  {intermediate_path}")
print(f"  • Registros: {len(df_reduced):,}")
print(f"  • Columnas: {len(df_reduced.columns)} (customer_id + is_premium + {len(features_retained)} features)")
print()
print(f"✓ Features eliminadas (CSV):")
print(f"  {METRICS_PATH}features_correlacion_eliminadas.csv")
print(f"  • Features: {len(to_drop)}")
print()

print("🎯 SIGUIENTE PASO:")
print("-" * 80)
print("Ejecutar: 06b_pca_and_save_artifacts")
print("   • Aplicará PCA para reducir dimensionalidad")
print(f"   • Reducirá de {len(features_retained)} features a ~12 componentes")
print("   • Guardará artefactos para inferencia")
print()
print("=" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Variables para la Parte 2
# MAGIC 
# MAGIC Las siguientes variables están listas para usar en la Parte 2:
# MAGIC - `X_reduced`: Features después de eliminar correlacionadas
# MAGIC - `y`: Target (is_premium)
# MAGIC - `customer_id`: IDs de clientes
# MAGIC - `features_retained`: Lista de features que pasaron el filtro
# MAGIC - `feature_cols`: Lista original de features

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06b_pca_and_save_artifacts (VERSIÓN SIMPLIFICADA)
# MAGIC 
# MAGIC **SOLUCIÓN PARA CLUSTERS SIN ACCESO AL SISTEMA DE ARCHIVOS LOCAL**
# MAGIC 
# MAGIC Esta versión NO usa archivos .pkl, en su lugar guarda:
# MAGIC - Parámetros del StandardScaler como CSV/Delta
# MAGIC - Componentes del PCA como CSV/Delta
# MAGIC - Estos se reconstruirán en inferencia

# COMMAND ----------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from datetime import datetime

GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
METRICS_PATH = "/Volumes/olist/olist_gold/metrics/"
MODELS_PATH = "/Volumes/olist/olist_gold/models/"

print("=" * 80)
print("🚀 PARTE 2: PCA Y GUARDADO DE ARTEFACTOS (VERSIÓN SIMPLIFICADA)")
print("=" * 80)
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0-5: (Igual que antes - cargar datos, estandarizar, aplicar PCA)

# COMMAND ----------

print("🔧 Verificando volume models...")
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.models")
    print("✅ Volume 'models' verificado/creado\n")
except Exception as e:
    print(f"⚠️  Volume models: {e}\n")

# COMMAND ----------

print("📥 Cargando dataset intermedio...")
intermediate_path = f"{GOLD_PATH}customer_features_rfm_20180930_after_correlation/"

try:
    df = spark.read.format("delta").load(intermediate_path).toPandas()
    print(f"✅ Dataset: {len(df):,} x {len(df.columns)}\n")
except Exception as e:
    print(f"❌ Error: {e}\n⚠️  Ejecuta primero: 06a_feature_correlation_reduction")
    raise

# COMMAND ----------

print("🎯 Preparando datos...")
customer_id = df['customer_id'].copy()
y = df['is_premium'].copy()
features_retained = [c for c in df.columns if c not in ['customer_id', 'is_premium']]
X = df[features_retained].copy()
print(f"✅ Clientes: {len(customer_id):,}, Features: {len(features_retained)}\n")

# COMMAND ----------

print("📏 ESTANDARIZACIÓN\n" + "="*80)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"✅ Shape: {X_scaled.shape}, Mean: {X_scaled.mean():.6f}, Std: {X_scaled.std():.6f}\n")

# COMMAND ----------

print("🔬 PCA AUTOMÁTICO\n" + "="*80)
pca_full = PCA()
pca_full.fit(X_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components = np.argmax(cumulative_variance >= 0.85) + 1
print(f"✅ Componentes: {n_components}, Varianza: {cumulative_variance[n_components-1]*100:.2f}%\n")

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)
print(f"✅ Transformación: {X_pca.shape}\n")

# COMMAND ----------

print("📦 Creando dataset final...")
pca_cols = [f'pca_{i+1}' for i in range(n_components)]
df_pca = pd.DataFrame(X_pca, columns=pca_cols)
df_pca['customer_id'] = customer_id.values
df_pca['is_premium'] = y.values
cols_order = ['customer_id', 'is_premium'] + pca_cols
df_pca = df_pca[cols_order]
print(f"✅ Dataset: {df_pca.shape}\n")

# COMMAND ----------

print("💾 Guardando dataset final...")
output_path = f"{GOLD_PATH}customer_features_rfm_20180930_reduced/"
spark.createDataFrame(df_pca).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(output_path)
print(f"✅ Guardado: {output_path}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. GUARDAR ARTEFACTOS (SIN PICKLE)

# COMMAND ----------

print("=" * 80)
print("💾 GUARDANDO ARTEFACTOS COMO CSV/DELTA")
print("=" * 80)
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8.1 Guardar Parámetros del StandardScaler

# COMMAND ----------

print("1️⃣ Guardando StandardScaler...")

# Crear DataFrame con parámetros
scaler_params = pd.DataFrame({
    'feature_name': features_retained,
    'feature_index': range(len(features_retained)),
    'mean': scaler.mean_,
    'scale': scaler.scale_,
    'var': scaler.var_
})

# Guardar en Delta
scaler_path = f"{MODELS_PATH}scaler_params/"
spark.createDataFrame(scaler_params).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(scaler_path)

print(f"✅ Scaler params guardados: {scaler_path}")
print(f"   Features: {len(features_retained)}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8.2 Guardar Componentes del PCA

# COMMAND ----------

print("2️⃣ Guardando PCA...")

# Guardar componentes principales (matriz de transformación)
pca_components = pd.DataFrame(
    pca.components_,
    columns=features_retained
)
pca_components['component_id'] = [f'PC{i+1}' for i in range(n_components)]

# Reordenar
cols = ['component_id'] + features_retained
pca_components = pca_components[cols]

# Guardar en Delta
pca_comp_path = f"{MODELS_PATH}pca_components/"
spark.createDataFrame(pca_components).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(pca_comp_path)

print(f"✅ PCA components guardados: {pca_comp_path}")
print(f"   Shape: {pca.components_.shape}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8.3 Guardar Parámetros del PCA

# COMMAND ----------

print("3️⃣ Guardando parámetros PCA...")

# Parámetros del PCA
pca_params = pd.DataFrame({
    'component_id': [f'PC{i+1}' for i in range(n_components)],
    'explained_variance': pca.explained_variance_,
    'explained_variance_ratio': pca.explained_variance_ratio_,
    'singular_values': pca.singular_values_
})

# Agregar mean_ del PCA (centro de los datos)
pca_mean = pd.DataFrame({
    'feature_name': features_retained,
    'pca_mean': pca.mean_
})

# Guardar parámetros
pca_params_path = f"{MODELS_PATH}pca_params/"
spark.createDataFrame(pca_params).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(pca_params_path)

pca_mean_path = f"{MODELS_PATH}pca_mean/"
spark.createDataFrame(pca_mean).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(pca_mean_path)

print(f"✅ PCA params guardados: {pca_params_path}")
print(f"✅ PCA mean guardado: {pca_mean_path}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8.4 Guardar lista de features retenidas

# COMMAND ----------

print("4️⃣ Guardando features retenidas...")

features_df = pd.DataFrame({
    'feature': features_retained,
    'order': range(len(features_retained))
})

features_path = f"{MODELS_PATH}features_retained/"
spark.createDataFrame(features_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(features_path)

print(f"✅ Features guardadas: {features_path}")
print(f"   Total: {len(features_retained)}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8.5 Guardar metadata

# COMMAND ----------

print("5️⃣ Guardando metadata...")

metadata = pd.DataFrame([{
    'n_features_original': len(features_retained),
    'n_features_after_correlation': len(features_retained),
    'n_pca_components': n_components,
    'pca_variance_explained': float(pca.explained_variance_ratio_.sum()),
    'correlation_threshold': 0.85,
    'training_date': datetime.now().strftime('%Y-%m-%d'),
    'cutoff_date': '2018-09-30 23:59:59',
    'scaler_type': 'StandardScaler',
    'pca_type': 'PCA',
    'storage_format': 'delta_tables'
}])

metadata_path = f"{MODELS_PATH}transformation_metadata/"
spark.createDataFrame(metadata).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").save(metadata_path)

print(f"✅ Metadata guardada: {metadata_path}")
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 8.6 Verificación

# COMMAND ----------

print("🔍 Verificando artefactos...")
print()

artifacts = {
    "scaler_params/": "StandardScaler parámetros",
    "pca_components/": "PCA componentes principales",
    "pca_params/": "PCA parámetros",
    "pca_mean/": "PCA mean",
    "features_retained/": "Features retenidas",
    "transformation_metadata/": "Metadata"
}

all_ok = True
for artifact, desc in artifacts.items():
    try:
        files = dbutils.fs.ls(f"{MODELS_PATH}{artifact}")
        print(f"✅ {artifact:30s} - {desc}")
    except:
        print(f"❌ {artifact:30s} - NO ENCONTRADO")
        all_ok = False

print()
if all_ok:
    print("✅ TODOS LOS ARTEFACTOS GUARDADOS EXITOSAMENTE")
else:
    print("❌ ALGUNOS ARTEFACTOS NO SE GUARDARON")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Visualizaciones

# COMMAND ----------

print("📈 Generando visualizaciones...")

fig = plt.figure(figsize=(15, 5))

# Scree Plot
ax1 = plt.subplot(131)
components_range = range(1, n_components + 1)
ax1.bar(components_range, pca.explained_variance_ratio_, alpha=0.7, color='steelblue')
ax1.plot(components_range, cumulative_variance[:n_components], 'ro-', linewidth=2)
ax1.axhline(y=0.85, color='red', linestyle='--', linewidth=2, label='85%')
ax1.set_xlabel('Componente')
ax1.set_ylabel('Varianza')
ax1.set_title('Scree Plot')
ax1.legend()
ax1.grid(alpha=0.3)

# Varianza acumulada
ax2 = plt.subplot(132)
ax2.plot(components_range, cumulative_variance[:n_components]*100, 'o-', linewidth=2)
ax2.axhline(y=85, color='red', linestyle='--', linewidth=2)
ax2.fill_between(components_range, 0, cumulative_variance[:n_components]*100, alpha=0.3)
ax2.set_xlabel('Componentes')
ax2.set_ylabel('Varianza Acumulada (%)')
ax2.set_title('Varianza Acumulada')
ax2.grid(alpha=0.3)

# Reducción
ax3 = plt.subplot(133)
stages = ['Después\nCorrelación', 'PCA']
dimensions = [len(features_retained), n_components]
bars = ax3.bar(stages, dimensions, color=['#f39c12', '#27ae60'], alpha=0.7, edgecolor='black')
ax3.set_ylabel('Dimensiones')
ax3.set_title('Reducción Dimensional')
for bar, dim in zip(bars, dimensions):
    ax3.text(bar.get_x() + bar.get_width()/2., dim, f'{int(dim)}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizaciones generadas\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Resumen Final

# COMMAND ----------

print()
print("=" * 80)
print("✅ PROCESO COMPLETADO")
print("=" * 80)
print()
print(f"📊 Features: {len(features_retained)} → {n_components} componentes PCA")
print(f"📊 Varianza: {cumulative_variance[n_components-1]*100:.2f}%")
print(f"📊 Reducción: {(1-n_components/len(features_retained))*100:.1f}%")
print()
print("💾 ARTEFACTOS GUARDADOS (formato Delta):")
print(f"   • scaler_params/")
print(f"   • pca_components/")
print(f"   • pca_params/")
print(f"   • pca_mean/")
print(f"   • features_retained/")
print(f"   • transformation_metadata/")
print()
print("🎯 SIGUIENTE PASO:")
print("   Usar notebook 10_inference_pipeline_production")
print("   (versión adaptada para cargar desde Delta)")
print()
print("=" * 80)